# Day A6 — S3·S4 객체와 Bioconductor의 자료 구조

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/A-r-bioconductor/day-A6.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/A-r-bioconductor/day-A6.md)

In [ ]:
# Colab 기본 R 런타임에 없는 Bioconductor 패키지 설치 (몇 분 걸립니다)
if (!requireNamespace("BiocManager", quietly = TRUE)) install.packages("BiocManager")
BiocManager::install(c("SummarizedExperiment"), update = FALSE, ask = FALSE)

## 예시로 이해하기

### 예시 1 — S3 클래스 하나 만들기

In [ ]:
new_panel <- function(counts, platform = "가상 패널") {
  stopifnot(is.matrix(counts), !is.null(colnames(counts)))
  structure(list(counts = counts, platform = platform), class = "panel_counts")
}
print.panel_counts <- function(x, ...) {
  cat("<panel_counts>", nrow(x$counts), "genes x", ncol(x$counts), "samples |", x$platform, "\n")
  invisible(x)
}
summary.panel_counts <- function(object, ...) colSums(object$counts)
m <- matrix(c(120, 80, 30, 200, 90, 25), nrow = 3,
            dimnames = list(c("Gfap", "Aif1", "Neg1"), c("S1", "S2")))
p <- new_panel(m)
p                       # print()가 자동으로 불린다 -> print.panel_counts
summary(p)
class(p); inherits(p, "panel_counts")
unclass(p)$platform
class(m); class(data.frame()); class(sum)

### 예시 2 — 행렬과 샘플 표를 함께 자르는 S4 클래스

In [ ]:
setClass("CountPanel",
         slots = c(counts = "matrix", colData = "data.frame"),
         validity = function(object) {
           if (!identical(colnames(object@counts), rownames(object@colData)))
             return("counts의 열 이름과 colData의 행 이름이 같은 순서로 일치해야 합니다")
           TRUE
         })
setGeneric("counts", function(x) standardGeneric("counts"))
setMethod("counts", "CountPanel", function(x) x@counts)
setMethod("show", "CountPanel", function(object)
  cat("CountPanel:", nrow(object@counts), "genes x", ncol(object@counts), "samples\n"))
setMethod("[", "CountPanel", function(x, i, j, ..., drop = FALSE) {
  if (missing(i)) i <- seq_len(nrow(x@counts))
  if (missing(j)) j <- seq_len(ncol(x@counts))
  new("CountPanel", counts = x@counts[i, j, drop = FALSE],
      colData = x@colData[j, , drop = FALSE])
})
m <- matrix(1:12, nrow = 3, dimnames = list(c("Gfap", "Aif1", "Cd68"), paste0("S", 1:4)))
cd <- data.frame(group = c("ctrl", "ctrl", "treat", "treat"), row.names = paste0("S", 1:4))
obj <- new("CountPanel", counts = m, colData = cd)
obj
sub <- obj[c("Gfap", "Cd68"), obj@colData$group == "treat"]
sub
counts(sub)
sub@colData
isVirtualClass("CountPanel"); is(obj, "CountPanel"); isS4(obj)
bad <- tryCatch(new("CountPanel", counts = m, colData = cd[4:1, , drop = FALSE]),
                error = function(e) conditionMessage(e))
cat(bad, "\n")
existsMethod("counts", "CountPanel")
showMethods("counts")

### 예시 3 — 진짜 SummarizedExperiment

> 설치는 처음 한 번만 하면 되고, 인터넷 연결과 몇 분(소스에서 컴파일하면 수십 분)이 필요합니다. 아래 출력은 R 4.3.2 + Bioconductor 3.18(SummarizedExperiment 설치 후)에서 "# 사용" 아래 줄을 실행한 그대로입니다. 패키지를 불러올 때 나오는 긴 "masked" 안내문은 뺐습니다.

In [ ]:
# 설치 (처음 한 번)
if (!requireNamespace("BiocManager", quietly = TRUE)) install.packages("BiocManager")
BiocManager::install("SummarizedExperiment")
BiocManager::version()          # 지금 R과 짝을 이룬 Bioconductor 릴리스

# 사용
library(SummarizedExperiment)
m  <- matrix(1:12, nrow = 3, dimnames = list(c("Gfap", "Aif1", "Cd68"), paste0("S", 1:4)))
cd <- DataFrame(group = c("ctrl", "ctrl", "treat", "treat"), row.names = paste0("S", 1:4))
rd <- DataFrame(probe_class = c("Endogenous", "Endogenous", "Endogenous"), row.names = rownames(m))
se <- SummarizedExperiment(assays = list(counts = m), colData = cd, rowData = rd)
se                                 # 요약 출력
assay(se, "counts")
colData(se)
se_treat <- se[, se$group == "treat"]   # assay와 colData가 함께 잘린다
dim(se_treat)
assay(se, "log2") <- log2(assay(se, "counts") + 1)   # 같은 그릇에 두 번째 assay 추가
assayNames(se)
browseVignettes("SummarizedExperiment")   # 브라우저에 공식 안내서가 열린다 (출력 생략)